# 01 — Frontend Audit

This notebook audits the landing page of termalne.obliview.com:
- Downloads and parses HTML
- Extracts all script/link/image assets
- Builds a source inventory
- Identifies candidate OGC endpoints in JavaScript

In [ ]:
from __future__ import annotations

import pathlib
import pandas as pd
from gdynia_thermal_audit.settings import Settings
from gdynia_thermal_audit.logging_utils import setup_logging
from gdynia_thermal_audit.frontend_audit.html_audit import audit_landing_page
from gdynia_thermal_audit.frontend_audit.js_inventory import inventory_scripts
from gdynia_thermal_audit.frontend_audit.asset_catalog import build_asset_catalog
from gdynia_thermal_audit.utils.io import ensure_dir
import httpx

setup_logging()
settings = Settings()

In [ ]:
OUTPUT_DIR = pathlib.Path(settings.output_dir)
RAW_DIR = pathlib.Path(settings.data_dir) / 'raw'
ensure_dir(OUTPUT_DIR)
ensure_dir(RAW_DIR)
print(f'Target URL: {settings.target_url}')

In [ ]:
# Run the landing-page audit
# Creates data/raw/landing_page.html and data/interim/asset_inventory.json
with httpx.Client(timeout=30, follow_redirects=True,
                  headers={'User-Agent': settings.user_agent}) as session:
    audit_results = audit_landing_page(
        url=settings.target_url,
        output_dir=RAW_DIR,
        session=session,
    )
print(f'Found {len(audit_results.get("scripts", []))} script tags')
print(f'Found {len(audit_results.get("links", []))} link tags')

In [ ]:
# Build asset catalog DataFrame
html_content = audit_results.get('html_content', '')
js_items = inventory_scripts(html_content)
df_catalog = build_asset_catalog(audit_results)
print(df_catalog.head())
df_catalog.to_csv(pathlib.Path(settings.data_dir) / 'interim' / 'asset_catalog.csv', index=False)

## Summary

Review `df_catalog` for candidate URLs with `asset_type` in `['wms','wmts','wfs','config']` — these are probed in notebook `02_network_probe.ipynb`.